# Lab 15: Explainability, Fairness, and Responsible Model Cards

            **Duration:** 3 hours  
            **Lecture alignment:** Week 15 — Explainable and responsible deep learning  
            **CLO mapping:** CLO-3, CLO-5  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Generate input-gradient and Grad-CAM explanations with native PyTorch.
- Audit demographic-parity, true-positive-rate, and false-positive-rate gaps on a labeled simulation.
- Evaluate a mitigation trade-off and document limitations in a model card.

            ## Three-hour activity plan

            - 0–35 min: train vision model and saliency
- 35–70 min: Grad-CAM and occlusion faithfulness
- 70–110 min: simulated fairness dataset/model
- 110–145 min: subgroup metrics and mitigation
- 145–180 min: model card, ethical limitations, and checks


## Book grounding

            - Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Thampi, *Interpretable AI*, Manning, 2022.
- Barocas, Hardt, and Narayanan, *Fairness and Machine Learning*, MIT Press, 2023.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20275
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_15")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_15"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 15, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Predict whether deleting the most salient pixels will reduce confidence more than deleting random pixels, and predict the fairness-gap direction created by the simulated label rule.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Native saliency and Grad-CAM on generated images


In [ ]:
def make_images(n,size=16):
    y=torch.randint(0,2,(n,));x=.05*torch.randn(n,1,size,size)
    for i,label in enumerate(y.tolist()):
        if label==0:x[i,0,2:-2,6:10]+=1
        else:x[i,0,6:10,2:-2]+=1
    return x.clamp(0,1),y
image_X,image_y=make_images(520 if FAST_MODE else 2400);cut=420 if FAST_MODE else 1900
class ExplainCNN(nn.Module):
    def __init__(self):super().__init__();self.conv1=nn.Conv2d(1,6,3,padding=1);self.conv2=nn.Conv2d(6,10,3,padding=1);self.head=nn.Linear(10,2)
    def forward(self,x):x=F.max_pool2d(F.relu(self.conv1(x)),2);x=F.relu(self.conv2(x));return self.head(F.adaptive_avg_pool2d(x,1).flatten(1))
vision=ExplainCNN().to(DEVICE);opt=torch.optim.Adam(vision.parameters(),lr=.012)
for _ in range(5 if FAST_MODE else 15):
    opt.zero_grad();loss=F.cross_entropy(vision(image_X[:cut].to(DEVICE)),image_y[:cut].to(DEVICE));loss.backward();opt.step()
sample=image_X[cut:cut+1].to(DEVICE).clone().requires_grad_(True);vision.zero_grad();logit=vision(sample);pred_class=logit.argmax(1).item();logit[0,pred_class].backward();saliency=sample.grad.abs().max(1).values.detach().cpu()[0]

activations={};gradients={}
h1=vision.conv2.register_forward_hook(lambda m,i,o:activations.__setitem__("value",o.detach()))
h2=vision.conv2.register_full_backward_hook(lambda m,gi,go:gradients.__setitem__("value",go[0].detach()))
vision.zero_grad();score=vision(image_X[cut:cut+1].to(DEVICE))[0,pred_class];score.backward();h1.remove();h2.remove()
weights=gradients["value"].mean((2,3),keepdim=True);cam=F.relu((weights*activations["value"]).sum(1,keepdim=True));cam=F.interpolate(cam,size=(16,16),mode="bilinear",align_corners=False)[0,0].cpu();cam=cam/(cam.max()+1e-8)

with torch.no_grad():base_conf=vision(image_X[cut:cut+1].to(DEVICE)).softmax(1)[0,pred_class].item()
flat=saliency.flatten();top=torch.topk(flat,int(.2*flat.numel())).indices;occluded=image_X[cut:cut+1].clone().flatten();occluded[top]=0;occluded=occluded.reshape(1,1,16,16)
with torch.no_grad():occluded_conf=vision(occluded.to(DEVICE)).softmax(1)[0,pred_class].item()
print({"prediction":pred_class,"base_confidence":base_conf,"top_saliency_occluded_confidence":occluded_conf})


## Activity 2 — Fairness audit on a pedagogical simulation


In [ ]:
print("IMPORTANT: The following fairness dataset is a pedagogical simulation. It is not evidence about any real demographic group.")
n=1200 if FAST_MODE else 5000;group=torch.randint(0,2,(n,));features=torch.randn(n,3)
latent=1.1*features[:,0]+.7*features[:,1]-.3*features[:,2]+.35*torch.randn(n)
outcome=(latent-.85*group>0).float()  # deliberately biased simulated labels
order=torch.randperm(n);tr=order[:int(.7*n)];te=order[int(.7*n):]
fair_model=nn.Sequential(nn.Linear(3,12),nn.ReLU(),nn.Linear(12,1)).to(DEVICE);opt=torch.optim.Adam(fair_model.parameters(),lr=.02)
for _ in range(45 if FAST_MODE else 130):
    opt.zero_grad();loss=F.binary_cross_entropy_with_logits(fair_model(features[tr].to(DEVICE)).squeeze(),outcome[tr].to(DEVICE));loss.backward();opt.step()
with torch.no_grad():probs=fair_model(features[te].to(DEVICE)).sigmoid().squeeze().cpu();truth=outcome[te];groups=group[te]
def audit(pred):
    rows={}
    for g in (0,1):
        m=groups==g;tp=((pred==1)&(truth==1)&m).sum().item();fn=((pred==0)&(truth==1)&m).sum().item();fp=((pred==1)&(truth==0)&m).sum().item();tn=((pred==0)&(truth==0)&m).sum().item()
        rows[g]={"selection_rate":pred[m].float().mean().item(),"tpr":tp/(tp+fn+1e-8),"fpr":fp/(fp+tn+1e-8)}
    rows["gaps"]={"demographic_parity":rows[1]["selection_rate"]-rows[0]["selection_rate"],"tpr":rows[1]["tpr"]-rows[0]["tpr"],"fpr":rows[1]["fpr"]-rows[0]["fpr"]}
    return rows
baseline_pred=(probs>=.5).long();baseline_audit=audit(baseline_pred)
best=None
for t0 in torch.linspace(.3,.7,9):
    for t1 in torch.linspace(.3,.7,9):
        pred=torch.where(groups==0,probs>=t0,probs>=t1).long();report=audit(pred);acc=(pred==truth).float().mean().item()
        objective=abs(report["gaps"]["tpr"])+.2*(1-acc)
        if best is None or objective<best[0]:best=(objective,float(t0),float(t1),pred,report,acc)
mitigated_audit=best[4];fairness_report={"simulation_notice":"Synthetic pedagogical simulation; not real-group evidence.","baseline":baseline_audit,"threshold_mitigation":mitigated_audit,"thresholds":{"group0":best[1],"group1":best[2]},"mitigated_accuracy":best[5]}
print(json.dumps(fairness_report,indent=2))


## Activity 3 — Evidence figures and model card


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(11,3.5));axes[0].imshow(image_X[cut,0],cmap="gray");axes[0].set_title("Input");axes[1].imshow(saliency,cmap="magma");axes[1].set_title("Input saliency");axes[2].imshow(cam,cmap="magma",vmin=0,vmax=1);axes[2].set_title("Grad-CAM")
for ax in axes:ax.axis("off")
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"xai_evidence.png",dpi=150);plt.show()
(ARTIFACT_DIR/"fairness_audit.json").write_text(json.dumps(fairness_report,indent=2))
model_card=(
    f"# Model Card — Pedagogical Simulation\n\n"
    f"Intended use: classroom study of XAI and fairness metrics only.\n\n"
    f"Data: entirely synthetic; protected-group values and biased outcomes are deliberately simulated.\n\n"
    f"XAI evidence: predicted class {pred_class}; confidence {base_conf:.3f}; after top-saliency occlusion {occluded_conf:.3f}.\n\n"
    f"Fairness: baseline TPR gap {baseline_audit['gaps']['tpr']:.3f}; post-threshold gap {mitigated_audit['gaps']['tpr']:.3f}.\n\n"
    "Limitations: saliency is not causal proof; group-specific thresholds may be inappropriate or unlawful in a real deployment; "
    "synthetic results must not be generalized to people.\n"
)
(ARTIFACT_DIR/"MODEL_CARD.md").write_text(model_card);print(model_card)


## Automated checks


In [ ]:
assert saliency.shape==(16,16) and cam.shape==(16,16) and torch.isfinite(cam).all()
assert set(groups.tolist())=={0,1} and fairness_report["simulation_notice"].startswith("Synthetic")
assert best[0] <= abs(baseline_audit["gaps"]["tpr"])+.2*(1-(baseline_pred==truth).float().mean().item())+1e-8
assert (ARTIFACT_DIR/"MODEL_CARD.md").exists() and (ARTIFACT_DIR/"fairness_audit.json").exists()
print("All Lab 15 checks passed.")


## Deliverables

                - Saliency/Grad-CAM evidence and occlusion result
- Fairness audit JSON with before/after trade-off
- Model card explicitly identifying simulated data and non-generalizability

                Submit the executed notebook and the files created in `/content/artifacts/lab_15/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    print("Extension: add a simulated second attribute and audit intersectional groups with minimum-support warnings.")
else:
    print("Extension disabled: intersectional audit or Integrated Gradients comparison.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
